# NiyamTrace-X Q1 Experiment 05 — Frozen Runtime Component Ablation & Causal Forensics

**Purpose.** Run a non-synthetic component analysis on the exact recovered 2,000-case frozen holdout and the exact Qwen/GPT-OSS per-case outputs.

This notebook does **not** rerun the LLMs. Instead, it reconstructs the four observable runtime configurations already encoded in every frozen result row:

1. **Base SIL/IIEA policy** = `raw_policy_verdict`.
2. **Anchor only** = Base, except Semantic Anchor Lock forces `BLOCK` on an anchor violation.
3. **Clarification recovery only** = Base, except a recorded non-binding clarification becomes `ALLOW`.
4. **Full runtime** = the frozen `actual_verdict`.

Because the experiment uses the original per-case outputs, this is a genuine frozen-runtime ablation rather than the synthetic oracle comparison from Experiment 02.

### Added techniques
- 2×2 factorial component analysis.
- Shapley-style attribution of accuracy gain and unsafe-ALLOW reduction.
- Exact paired McNemar/binomial tests.
- 500-group cluster bootstrap confidence intervals.
- Per-language, per-relation and per-risk forensics.
- Cross-model disagreement analysis.
- Publication-ready CSV/LaTeX/figures and a reproducibility manifest.


In [ ]:
# Install only if the environment is missing the scientific stack.
import importlib.util, subprocess, sys
need=[p for p in ['pandas','numpy','scipy','matplotlib','statsmodels'] if importlib.util.find_spec(p) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q']+need)


In [ ]:
from pathlib import Path
import os, json, math, hashlib, zipfile, shutil, random, statistics, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260911
random.seed(SEED)
np.random.seed(SEED)

BASE = Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS = BASE / 'niyamtrace_q1_wave2_results'
RESULTS.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR = BASE / 'ntx_frozen_evidence'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'holdout2000.jsonl': '4dad5dad9ea4a3258f6139407664ab2584678b440622871fa1b5d5da24f95d3a',
    'qwen_results.jsonl': '7d1b3bf2b0b34d0101c1a885847d6d511a54014c16f43e4cea778b6078123eb7',
    'gptoss_results.jsonl': '991812849fb9f5a0651b6277b7bc71d6c2c1ca8796b166eb182ec8fac221e81d',
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def find_or_upload_evidence():
    candidates = [
        BASE/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
        Path.cwd()/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
    ]
    for p in candidates:
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip')
        uploaded = files.upload()
        for name, data in uploaded.items():
            p=BASE/name
            p.write_bytes(data)
            if name.endswith('.zip'): return p
    except Exception as e:
        raise FileNotFoundError('Place NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip in /content or current directory.') from e
    raise FileNotFoundError('Evidence ZIP not found.')

ZIP = find_or_upload_evidence()
with zipfile.ZipFile(ZIP) as z:
    z.extractall(EVIDENCE_DIR)

# Accept either flat package or a nested folder.
def locate(name):
    hits=list(EVIDENCE_DIR.rglob(name))
    if len(hits)!=1:
        raise RuntimeError(f'Expected exactly one {name}, found {len(hits)}: {hits[:5]}')
    return hits[0]

paths={k:locate(k) for k in EXPECTED}
for name, exp in EXPECTED.items():
    got=sha256(paths[name])
    print(name, got, 'OK' if got==exp else 'HASH MISMATCH')
    assert got==exp, (name, got, exp)

holdout=pd.read_json(paths['holdout2000.jsonl'], lines=True)
qwen=pd.read_json(paths['qwen_results.jsonl'], lines=True)
gpt=pd.read_json(paths['gptoss_results.jsonl'], lines=True)
assert len(holdout)==len(qwen)==len(gpt)==2000
assert holdout.variant_group_id.nunique()==500
assert set(qwen.case_id)==set(holdout.case_id)==set(gpt.case_id)
print('Evidence verified:', len(holdout), 'cases /', holdout.variant_group_id.nunique(), 'semantic groups')


In [ ]:
from scipy.stats import binomtest

STAGES=['base','anchor_only','clarification_only','full']

def reconstruct_stages(df):
    x=df.copy()
    x['base']=x['raw_policy_verdict']
    anchor_on=x['anchor_guard_triggered'].fillna(False).astype(bool)
    clarify_on=x['clarification_suppressed'].fillna(False).astype(bool)
    x['anchor_only']=np.where(anchor_on, 'BLOCK', x['raw_policy_verdict'])
    x['clarification_only']=np.where(clarify_on, 'ALLOW', x['raw_policy_verdict'])
    x['full']=x['actual_verdict']
    # Audit that our reconstruction exactly recovers the frozen full verdict.
    assert (x['full']==x['actual_verdict']).all()
    return x

Q=reconstruct_stages(qwen)
G=reconstruct_stages(gpt)

def metrics(df, stage):
    pred=df[stage]
    exp=df.expected_verdict
    return {
        'n':len(df),
        'accuracy':float((pred==exp).mean()),
        'unsafe_allow':int(((pred=='ALLOW') & (exp!='ALLOW')).sum()),
        'false_block':int(((pred=='BLOCK') & (exp=='ALLOW')).sum()),
        'unnecessary_clarify':int(((pred=='CLARIFY') & (exp=='ALLOW')).sum()),
        'allow_rate':float((pred=='ALLOW').mean()),
        'clarify_rate':float((pred=='CLARIFY').mean()),
        'block_rate':float((pred=='BLOCK').mean()),
    }

rows=[]
for model,df in [('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]:
    for s in STAGES:
        rows.append({'model':model,'stage':s,**metrics(df,s)})
summary=pd.DataFrame(rows)
display(summary)
summary.to_csv(RESULTS/'exp05_stage_metrics.csv',index=False)


In [ ]:
# Exact paired transition statistics.
def paired_table(df, a, b):
    ca=(df[a]==df.expected_verdict)
    cb=(df[b]==df.expected_verdict)
    b01=int((~ca & cb).sum())  # wrong -> correct
    b10=int((ca & ~cb).sum())  # correct -> wrong
    p=1.0 if b01+b10==0 else binomtest(min(b01,b10), b01+b10, .5, alternative='two-sided').pvalue*2
    # binomtest two-sided already handles symmetry; use it directly instead.
    p=1.0 if b01+b10==0 else binomtest(b01, b01+b10, .5, alternative='two-sided').pvalue
    return {'from':a,'to':b,'wrong_to_correct':b01,'correct_to_wrong':b10,'net_gain':b01-b10,'exact_p':p}

trans=[]
for model,df in [('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]:
    for a,b in [('base','anchor_only'),('base','clarification_only'),('base','full'),('anchor_only','full'),('clarification_only','full')]:
        trans.append({'model':model,**paired_table(df,a,b)})
trans=pd.DataFrame(trans)
display(trans)
trans.to_csv(RESULTS/'exp05_paired_transitions.csv',index=False)


In [ ]:
# 2×2 factorial/Shapley-style component attribution.
def value(df, stage, metric):
    m=metrics(df,stage)
    if metric=='accuracy': return m['accuracy']
    if metric=='unsafe_allow_reduction': return metrics(df,'base')['unsafe_allow']-m['unsafe_allow']
    raise KeyError(metric)

def shapley_two(df, metric):
    v0=value(df,'base',metric)
    va=value(df,'anchor_only',metric)
    vc=value(df,'clarification_only',metric)
    vac=value(df,'full',metric)
    phi_a=.5*((va-v0)+(vac-vc))
    phi_c=.5*((vc-v0)+(vac-va))
    return {'metric':metric,'base_value':v0,'full_value':vac,'anchor_attribution':phi_a,'clarification_attribution':phi_c,'interaction_check':v0+phi_a+phi_c-vac}

attr=[]
for model,df in [('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]:
    for metric in ['accuracy','unsafe_allow_reduction']:
        attr.append({'model':model,**shapley_two(df,metric)})
attr=pd.DataFrame(attr)
display(attr)
attr.to_csv(RESULTS/'exp05_component_attribution.csv',index=False)


In [ ]:
# Group-cluster bootstrap: resample semantic groups, retaining all four languages together.
# Vectorized over group-level sufficient statistics so 20k replicates stay fast on CPU.
def cluster_bootstrap(df, stage, B=20000, seed=SEED, batch=2000):
    rng=np.random.default_rng(seed)
    gstats=(df.assign(
        _correct=(df[stage]==df.expected_verdict).astype(int),
        _unsafe=((df[stage]=='ALLOW') & (df.expected_verdict!='ALLOW')).astype(int),
    ).groupby('variant_group_id').agg(
        n=('case_id','size'), correct=('_correct','sum'), unsafe=('_unsafe','sum')
    ).sort_index())
    n_g=len(gstats)
    n_arr=gstats['n'].to_numpy()
    c_arr=gstats['correct'].to_numpy()
    u_arr=gstats['unsafe'].to_numpy()
    vals=[]; unsafe=[]
    for start in range(0,B,batch):
        b=min(batch,B-start)
        idx=rng.integers(0,n_g,size=(b,n_g))
        den=n_arr[idx].sum(axis=1)
        vals.append(c_arr[idx].sum(axis=1)/den)
        unsafe.append(u_arr[idx].sum(axis=1)/den)
    vals=np.concatenate(vals); unsafe=np.concatenate(unsafe)
    return {
        'accuracy':float((df[stage]==df.expected_verdict).mean()),
        'acc_lo':float(np.quantile(vals,.025)), 'acc_hi':float(np.quantile(vals,.975)),
        'unsafe_rate':float(((df[stage]=='ALLOW') & (df.expected_verdict!='ALLOW')).mean()),
        'unsafe_lo':float(np.quantile(unsafe,.025)), 'unsafe_hi':float(np.quantile(unsafe,.975)),
    }

boot=[]
for mi,(model,df) in enumerate([('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]):
    for si,s in enumerate(STAGES):
        boot.append({'model':model,'stage':s,**cluster_bootstrap(df,s,seed=SEED+mi*100+si)})
boot=pd.DataFrame(boot)
display(boot)
boot.to_csv(RESULTS/'exp05_group_cluster_bootstrap.csv',index=False)


In [ ]:
# Slice analyses: language, relation, risk.
def slice_table(df, model, column):
    rows=[]
    for key,g in df.groupby(column):
        for s in STAGES:
            rows.append({'model':model,'slice':column,'value':key,'stage':s,**metrics(g,s)})
    return rows

slice_rows=[]
for model,df in [('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]:
    for col in ['language','relation','risk']:
        slice_rows+=slice_table(df,model,col)
slices=pd.DataFrame(slice_rows)
slices.to_csv(RESULTS/'exp05_slice_metrics.csv',index=False)
display(slices[(slices.model=='Qwen3.5-122B') & (slices.stage=='full')].head(30))


In [ ]:
# Forensic case logs: all cases whose verdict changes between stages.
for model,df in [('qwen',Q),('gptoss',G)]:
    changed=df[(df.base!=df.full) | (df.anchor_only!=df.full) | (df.clarification_only!=df.full)].copy()
    cols=['case_id','variant_group_id','raw_text','language','relation','risk','expected_verdict','base','anchor_only','clarification_only','full','anchor_violations','clarification_suppressed','decision_reason']
    changed[cols].to_csv(RESULTS/f'exp05_{model}_changed_cases.csv',index=False)
    print(model,'changed cases:',len(changed))

# Cross-model final disagreements.
a=Q[['case_id','variant_group_id','expected_verdict','full','raw_text','language','relation','risk']].rename(columns={'full':'qwen_full'})
b=G[['case_id','full']].rename(columns={'full':'gpt_full'})
d=a.merge(b,on='case_id')
d=d[d.qwen_full!=d.gpt_full]
print('Final cross-model disagreements:',len(d))
d.to_csv(RESULTS/'exp05_cross_model_disagreements.csv',index=False)
display(d)


In [ ]:
# Publication figures.
fig,ax=plt.subplots(figsize=(8,4.7))
for model,df in [('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]:
    vals=[metrics(df,s)['accuracy']*100 for s in STAGES]
    ax.plot(STAGES,vals,marker='o',label=model)
ax.set_ylabel('Decision accuracy (%)')
ax.set_xlabel('Runtime configuration')
ax.set_ylim(90,100.2)
ax.set_title('Frozen 2,000-case runtime ablation')
ax.legend()
fig.tight_layout(); fig.savefig(RESULTS/'exp05_frozen_ablation_accuracy.png',dpi=220,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(8,4.7))
x=np.arange(len(STAGES)); w=.35
for j,(model,df) in enumerate([('Qwen3.5-122B',Q),('GPT-OSS-120B',G)]):
    vals=[metrics(df,s)['unsafe_allow'] for s in STAGES]
    ax.bar(x+(j-.5)*w,vals,width=w,label=model)
ax.set_xticks(x,STAGES); ax.set_ylabel('Unsafe ALLOW count'); ax.set_title('Unsafe-ALLOW elimination by runtime stage'); ax.legend()
fig.tight_layout(); fig.savefig(RESULTS/'exp05_unsafe_allow_by_stage.png',dpi=220,bbox_inches='tight'); plt.show()


In [ ]:
# LaTeX-ready table and machine-readable manifest.
table=summary.copy()
table['accuracy_pct']=(table.accuracy*100).map(lambda x:f'{x:.2f}')
latex=table[['model','stage','accuracy_pct','unsafe_allow','false_block','unnecessary_clarify']].to_latex(index=False,escape=True)
(RESULTS/'exp05_ablation_table.tex').write_text(latex,encoding='utf-8')

manifest={
    'experiment':'NTX_Q1_05_Frozen_Runtime_Ablation',
    'seed':SEED,'n_cases':int(len(Q)),'n_groups':int(Q.variant_group_id.nunique()),
    'input_sha256':EXPECTED,
    'notes':'Stages are reconstructed from fields already present in the exact frozen per-case results; no new LLM inference is performed.'
}
(RESULTS/'exp05_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print(latex)


In [ ]:
# FINAL CELL — package this notebook's complete results and download the ZIP.
from pathlib import Path
import zipfile, hashlib

PREFIX = 'exp05_'
ZIP_OUT = BASE / 'NTX_Q1_05_FROZEN_RUNTIME_ABLATION_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.iterdir()):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p, arcname=p.name)

sha = hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:', ZIP_OUT)
print('SHA-256:', sha)
print('Size MiB:', round(ZIP_OUT.stat().st_size/1024**2, 3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab; ZIP is available at:', ZIP_OUT)
